In [2]:
import pandas as pd
df = pd.read_csv(    "../data/raw/customer_support_tickets.csv")
df2=pd.read_csv("../data/processed/pseudo_labels.csv")
print(df.shape)
df.head()

(20000, 12)


,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,Elena Rodriguez,5
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,Anya Sharma,5
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,David Kim,5


In [3]:
df2.head()

,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,...,full_text,llm_score,cluster_score,resolution_score,rule_score,rule_evidence,inferred_severity,severity_delta,mismatch,mismatch_type
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",General Inquiry,High,Web Form,2025-07-02,43,...,"hours of operation - individual hi support, wh...",0,1,1,0,NaN,Medium,-1,1,False Alarm
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,...,"data not syncing - card hi support, the applic...",2,1,1,3,crash,High,0,0,Consistent
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,...,"2fa issues - question hi support, how do i upg...",0,0,3,0,NaN,Medium,-1,1,False Alarm
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,...,"login failed - let hi support, the dashboard i...",2,1,1,3,login failed,High,2,1,Hidden Crisis
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,...,"refund status - attention hi support, i have b...",1,1,1,2,refund,Medium,0,0,Consistent


In [5]:
print(
    pd.crosstab(
        df2["Priority_Level"],
        df2["inferred_severity"]
    )
)

inferred_severity  Critical  High   Low  Medium
Priority_Level                                 
Critical                164   952     1     181
High                     46  1312   424    1634
Low                       0   720  3088    3908
Medium                    0  1091  2411    4068


In [6]:
print(df.columns.tolist())

['Ticket_ID', 'Customer_Name', 'Customer_Email', 'Ticket_Subject', 'Ticket_Description', 'Issue_Category', 'Priority_Level', 'Ticket_Channel', 'Submission_Date', 'Resolution_Time_Hours', 'Assigned_Agent', 'Satisfaction_Score']


In [7]:
print(df.shape)

print("\nPriority Distribution")
print(df["Priority_Level"].value_counts())

print("\nIssue Categories")
print(df["Issue_Category"].value_counts())

print("\nResolution Stats")
print(df["Resolution_Time_Hours"].describe())

(20000, 12)

Priority Distribution
Priority_Level
Low         7716
Medium      7570
High        3416
Critical    1298
Name: count, dtype: int64

Issue Categories
Issue_Category
Technical          5918
Billing            5036
Account            4081
General Inquiry    3925
Fraud              1040
Name: count, dtype: int64

Resolution Stats
count    20000.000000
mean        39.230300
std         35.221884
min          1.000000
25%         11.000000
50%         27.000000
75%         58.000000
max        120.000000
Name: Resolution_Time_Hours, dtype: float64


In [14]:
print(
    df2["cluster_score"]
    .value_counts()
    .sort_index()
)

print(
    df2["resolution_score"]
    .value_counts()
    .sort_index()
)

print(
    df2["rule_score"]
    .value_counts()
    .sort_index()
)

cluster_score
0     7242
1    11719
3     1039
Name: count, dtype: int64
resolution_score
0    6567
1    3893
2    4041
3    5499
Name: count, dtype: int64
rule_score
0    13119
2     2069
3     4812
Name: count, dtype: int64


In [11]:
sample = df[
    [
        "Ticket_ID",
        "Ticket_Subject",
        "Ticket_Description"
    ]
]

sample.to_csv(
    "llm_input.csv",
    index=False
)

In [12]:
import pandas as pd
for col in [
    "Issue_Category",
    "Ticket_Channel"
]:
    print("\n", col)
    print(df[col].value_counts())


 Issue_Category
Issue_Category
Technical          5918
Billing            5036
Account            4081
General Inquiry    3925
Fraud              1040
Name: count, dtype: int64

 Ticket_Channel
Ticket_Channel
Chat        6693
Email       6656
Web Form    6651
Name: count, dtype: int64


In [13]:
print(
    df.groupby(
        "Priority_Level"
    )["Resolution_Time_Hours"]
    .mean()
)

Priority_Level
Critical    12.068567
High        24.520492
Low         45.168351
Medium      44.472919
Name: Resolution_Time_Hours, dtype: float64


In [16]:
print(df2["mismatch"].value_counts())

print(df2["inferred_severity"].value_counts())

print(
    pd.crosstab(
        df2["Priority_Level"],
        df2["inferred_severity"]
    )
)

mismatch
1    11368
0     8632
Name: count, dtype: int64
inferred_severity
Medium      9791
Low         5924
High        4075
Critical     210
Name: count, dtype: int64
inferred_severity  Critical  High   Low  Medium
Priority_Level                                 
Critical                164   952     1     181
High                     46  1312   424    1634
Low                       0   720  3088    3908
Medium                    0  1091  2411    4068


In [17]:
df2 = pd.read_csv(
    "../data/processed/pseudo_labels.csv"
)

print(
    df2["mismatch"]
    .value_counts(normalize=True)
)

mismatch
1    0.5684
0    0.4316
Name: proportion, dtype: float64


In [ ]:
import sys
sys.path.append("../src")
from models.create_dataset import create_splits 
create_splits(
    "../data/processed/pseudo_labels.csv",
    "../data/processed"
)

In [32]:
import pandas as pd

train = pd.read_csv(
    "../data/processed/train.csv"
)

val = pd.read_csv(
    "../data/processed/val.csv"
)

test = pd.read_csv(
    "../data/processed/test.csv"
)

print(train.shape)
print(val.shape)
print(test.shape)

(16000, 23)
(2000, 23)
(2000, 23)


In [19]:
print(
    pd.crosstab(
        df2["Priority_Level"],
        df2["inferred_severity"],
        normalize="index"
    )
)

inferred_severity  Critical      High       Low    Medium
Priority_Level                                           
Critical           0.126348  0.733436  0.000770  0.139445
High               0.013466  0.384075  0.124122  0.478337
Low                0.000000  0.093313  0.400207  0.506480
Medium             0.000000  0.144122  0.318494  0.537384


In [21]:
print((df2["llm_score"] == df2["cluster_score"]).mean())
print((df2["llm_score"] == df2["resolution_score"]).mean())
print((df2["cluster_score"] == df2["resolution_score"]).mean())

0.36075
0.3066
0.28175


In [22]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/pseudo_labels.csv"
)

print(
    (df["llm_score"] == df["cluster_score"]).mean()
)

print(
    (df["llm_score"] == df["resolution_score"]).mean()
)

print(
    (df["cluster_score"] == df["resolution_score"]).mean()
)

print(
    df["mismatch"].value_counts(normalize=True)
)

print(
    pd.crosstab(
        df["Priority_Level"],
        df["inferred_severity"],
        normalize="index"
    )
)

0.36075
0.3066
0.28175
mismatch
1    0.5684
0    0.4316
Name: proportion, dtype: float64
inferred_severity  Critical      High       Low    Medium
Priority_Level                                           
Critical           0.126348  0.733436  0.000770  0.139445
High               0.013466  0.384075  0.124122  0.478337
Low                0.000000  0.093313  0.400207  0.506480
Medium             0.000000  0.144122  0.318494  0.537384


In [2]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

df = pd.read_csv("../data/processed/pseudo_labels.csv")

SEVERITY_MAP = {"Low": 0, "Medium": 1, "High": 2, "Critical": 3}

assigned = df["Priority_Level"].map(SEVERITY_MAP)
fused    = df["inferred_severity"].map(SEVERITY_MAP)

# Binary mismatch per signal (1 if signal disagrees with assigned priority)
signals = {
    "LLM":        (df["llm_score"]        != assigned).astype(int),
    "Cluster":    (df["cluster_score"]    != assigned).astype(int),
    "Resolution": (df["resolution_score"] != assigned).astype(int),
    "Rule":       (df["rule_score"]       != assigned).astype(int),
}

fused_mismatch = df["mismatch"]  # already binary (0/1)

print("Pairwise Agreement with Fused Label:\n")
for name, signal_mismatch in signals.items():
    agreement = (signal_mismatch == fused_mismatch).mean()
    kappa     = cohen_kappa_score(signal_mismatch, fused_mismatch)
    print(f"  {name:<12} Agreement: {agreement:.3f}   Cohen's κ: {kappa:.3f}")

# Pairwise between signals themselves
print("\nPairwise Agreement Between Signals:\n")
names = list(signals.keys())
for i in range(len(names)):
    for j in range(i+1, len(names)):
        a, b = signals[names[i]], signals[names[j]]
        agreement = (a == b).mean()
        print(f"  {names[i]} vs {names[j]:<12} Agreement: {agreement:.3f}")

Pairwise Agreement with Fused Label:

  LLM          Agreement: 0.633   Cohen's κ: 0.243
  Cluster      Agreement: 0.660   Cohen's κ: 0.313
  Resolution   Agreement: 0.610   Cohen's κ: 0.178
  Rule         Agreement: 0.547   Cohen's κ: 0.047

Pairwise Agreement Between Signals:

  LLM vs Cluster      Agreement: 0.519
  LLM vs Resolution   Agreement: 0.585
  LLM vs Rule         Agreement: 0.759
  Cluster vs Resolution   Agreement: 0.520
  Cluster vs Rule         Agreement: 0.533
  Resolution vs Rule         Agreement: 0.625
